# 36. Data Engineering for AI

**Tier:** Frontier
**Estimated time:** 50 minutes
**Prerequisites:** 14, 15, 25
**Priority:** 🟡 Important — this is the unfair advantage: the Data-Engineering × AI intersection almost no curriculum teaches, and it converts existing expertise into differentiation rather than adding a brand-new skill. *If skipped, revisit when:* building any RAG corpus at scale, or when asked to generate synthetic training/eval data.
**Source material:** notebooks 14/15 (RAG fundamentals, vector DBs); notebook 25 (benchmark hygiene, synthetic-data contamination)

## What You'll Learn
- Embedding pipelines at scale: batching and incremental refresh instead of re-embedding everything every time
- Data quality for RAG corpora: deduplication and chunk QA before anything reaches a vector store
- Generating synthetic training/eval data safely, with the contamination discipline from notebook 25 built in from the start
- Schema design for storing traces/conversations so they're actually queryable later

## Why This Matters
Everything in Tiers 3-4 assumes clean, well-chunked, deduplicated data arrives at the embedding step. In practice, a Data Engineer's existing skills — batch pipeline design, incremental processing, schema design, data quality gates — are exactly what's missing from most AI engineers' toolkits, and exactly what turns a working RAG demo into a system that survives contact with a real, messy, growing corpus. This notebook is where your background becomes your edge instead of something to work around.


In [ ]:
import os, hashlib, json
import numpy as np

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
TEACH_MODEL = "claude-haiku-4-5-20251001"

if HAS_ANTHROPIC:
    import anthropic
    client = anthropic.Anthropic()
    print("Anthropic ready.")
else:
    client = None
    print("No ANTHROPIC_API_KEY — synthetic-data generation cells will be skipped.")


## Embedding pipelines at scale: batching and incremental refresh

Notebook 15 embedded a handful of documents in one shot. A real corpus grows continuously — re-embedding everything on every update wastes compute linearly with corpus size. The standard fix, straight out of data engineering: batch new/changed documents, and use a content hash to skip anything that hasn't actually changed since the last run.

In [ ]:
def content_hash(text):
    return hashlib.sha256(text.encode()).hexdigest()[:16]

class IncrementalEmbeddingIndex:
    """Tracks which documents have already been embedded, keyed by content hash, so an
    update pass only re-embeds what actually changed (or is new)."""
    def __init__(self):
        self._hash_to_doc_id = {}   # content_hash -> stable doc id
        self._embedded = set()       # doc ids already embedded

    def sync(self, documents: dict):
        """documents: {doc_id: text}. Returns (to_embed, unchanged, deleted)."""
        to_embed, unchanged = [], []
        current_ids = set(documents.keys())
        for doc_id, text in documents.items():
            h = content_hash(text)
            if self._hash_to_doc_id.get(doc_id) == h:
                unchanged.append(doc_id)
            else:
                to_embed.append(doc_id)
                self._hash_to_doc_id[doc_id] = h
                self._embedded.add(doc_id)
        deleted = self._embedded - current_ids
        self._embedded -= deleted
        return to_embed, unchanged, list(deleted)

index = IncrementalEmbeddingIndex()

corpus_v1 = {"doc1": "The KV cache speeds up decoding.", "doc2": "Attention weighs token relevance."}
to_embed, unchanged, deleted = index.sync(corpus_v1)
print(f"First sync: embed={to_embed}, unchanged={unchanged}, deleted={deleted}")

# Second sync: doc1 unchanged, doc2 edited, doc3 is brand new.
corpus_v2 = {"doc1": "The KV cache speeds up decoding.",
             "doc2": "Attention weighs token relevance across the WHOLE sequence.",
             "doc3": "LoRA fine-tunes a small set of adapter weights."}
to_embed, unchanged, deleted = index.sync(corpus_v2)
print(f"Second sync: embed={to_embed}, unchanged={unchanged}, deleted={deleted}")
print(f"\n-> Only {len(to_embed)} of {len(corpus_v2)} documents needed re-embedding, not all {len(corpus_v2)}.")


## Data quality for RAG corpora: dedup and chunk QA

Before anything reaches a vector store (notebook 15), a data-quality pass catches two common corpus problems: near-duplicate documents that waste index space and skew retrieval toward over-represented content, and chunks that are too small/large or badly split (mid-sentence) to be useful retrieval units. Both are exactly the kind of validation-before-load discipline a data pipeline already enforces for any other data source.

In [ ]:
import difflib

def find_near_duplicates(documents: dict, threshold=0.85):
    """Flag document pairs that are near-duplicates, so only one survives into the index."""
    ids = list(documents.keys())
    duplicates = []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            ratio = difflib.SequenceMatcher(None, documents[ids[i]], documents[ids[j]]).ratio()
            if ratio >= threshold:
                duplicates.append((ids[i], ids[j], round(ratio, 2)))
    return duplicates

def chunk_quality_check(chunk_text, min_chars=40, max_chars=500):
    issues = []
    if len(chunk_text) < min_chars:
        issues.append("too short — likely uninformative on its own")
    if len(chunk_text) > max_chars:
        issues.append("too long — hurts retrieval precision, should be split")
    if chunk_text and chunk_text[0].islower() and not chunk_text.startswith(("e.g.", "i.e.")):
        issues.append("starts mid-sentence — likely a bad split boundary")
    if chunk_text and chunk_text.strip()[-1] not in ".!?\"'":
        issues.append("ends mid-sentence — likely a bad split boundary")
    return issues

messy_corpus = {
    "a": "The KV cache stores past attention keys and values so decoding does not recompute them.",
    "b": "The KV cache stores past keys and values so decoding doesn't need to recompute them.",  # near-dup of a
    "c": "attention lets the model weigh how relevant each token is to every other",   # bad boundaries + no key term
    "d": "ok",   # too short
}

print("Near-duplicates:", find_near_duplicates(messy_corpus))
for doc_id, text in messy_corpus.items():
    issues = chunk_quality_check(text)
    if issues:
        print(f"[{doc_id}] {text[:50]!r} -> {issues}")


## Generating synthetic data — with contamination discipline built in

Synthetic data generation (using an LLM to create training or eval examples) is a genuine data-engineering skill applied to a new kind of "data source." Notebook 25 named synthetic-data contamination as pathway #5 — the risk that the SAME model (or one that shares training data) grading or generating your eval set has effectively seen it already. The discipline: generate FROM a clearly separate seed set, and scan the output against your real eval set with the exact near-duplicate check from notebook 25 before anything gets used.

In [ ]:
def generate_synthetic_qa(topic, n=3):
    if not HAS_ANTHROPIC:
        return []
    prompt = (f"Generate {n} short factual question-answer pairs about {topic}. "
              "Format each as 'Q: ...\nA: ...' with a blank line between pairs.")
    raw = client.messages.create(model=TEACH_MODEL, max_tokens=300,
                                  messages=[{"role": "user", "content": prompt}]).content[0].text
    pairs = []
    for block in raw.split("\n\n"):
        if block.strip().startswith("Q:"):
            lines = block.strip().splitlines()
            q = lines[0][2:].strip()
            a = lines[1][2:].strip() if len(lines) > 1 and lines[1].startswith("A:") else ""
            if q and a:
                pairs.append({"q": q, "a": a})
    return pairs

# The REAL eval set this synthetic data must never leak into (same shape as notebook 24's GOLDEN).
REAL_EVAL_SET = [
    {"q": "What is the capital of France?", "a": "Paris"},
    {"q": "What does a KV cache store?", "a": "Past attention keys and values"},
]

synthetic = generate_synthetic_qa("world capitals", n=3)
print("Generated synthetic examples:")
for s in synthetic:
    print(" ", s)

# Contamination check (notebook 25's near-duplicate scanner, reused unmodified in spirit).
def near_duplicate_ratio(a, b):
    return difflib.SequenceMatcher(None, a.lower(), b.lower()).ratio()

print("\nContamination scan against REAL_EVAL_SET:")
for s in synthetic:
    best = max(REAL_EVAL_SET, key=lambda r: near_duplicate_ratio(s["q"], r["q"]))
    ratio = near_duplicate_ratio(s["q"], best["q"])
    flag = "CONTAMINATED — discard" if ratio >= 0.7 else "clean"
    print(f"  [{flag}] ratio={ratio:.2f}  synthetic={s['q']!r}")


## Schema design for trace/conversation storage

Notebook 27 traced production requests; that data is only useful later if it was stored with a schema that supports the queries you'll actually want to run — "show me every low-score interaction from last week," "how did cost per request change after the prompt update." This is a completely ordinary data-modeling problem, just applied to conversation traces instead of orders or events.

In [ ]:
TRACE_SCHEMA_EXAMPLE = {
    "trace_id": "uuid, primary key",
    "session_id": "uuid, groups multi-turn conversations — enables 'show me this whole session'",
    "timestamp": "ISO8601, indexed — enables time-range queries and drift analysis (notebook 27)",
    "model": "string, indexed — enables 'compare model versions' queries",
    "prompt_version": "string, indexed — enables the CI regression-gate queries from notebook 33",
    "input_tokens": "int",
    "output_tokens": "int",
    "cost_usd": "float, indexed — enables cost-over-time dashboards (notebook 32)",
    "latency_ms": "int, indexed — enables p50/p95/p99 latency dashboards",
    "eval_score": "float, nullable — populated by an offline eval job, NOT at request time",
    "user_feedback": "enum(thumbs_up, thumbs_down, null) — the feedback loop from notebook 27",
}

def validate_trace_record(record: dict):
    missing = [field for field in TRACE_SCHEMA_EXAMPLE if field not in record]
    return {"valid": not missing, "missing_fields": missing}

good_record = {
    "trace_id": "abc123", "session_id": "sess1", "timestamp": "2026-07-05T10:00:00Z",
    "model": "claude-haiku-4-5", "prompt_version": "v3", "input_tokens": 120, "output_tokens": 40,
    "cost_usd": 0.0004, "latency_ms": 850, "eval_score": None, "user_feedback": None,
}
print(validate_trace_record(good_record))
print(validate_trace_record({"trace_id": "abc124"}))


## Exercises

**Exercise 1 (Warm-up):** Add a third sync (`corpus_v3`) to the `IncrementalEmbeddingIndex` demo that deletes `doc1` entirely, and confirm `sync()` reports it correctly in the `deleted` list.

**Exercise 2 (Apply):** Implement `dedup_corpus(documents, threshold=0.85) -> dict` that uses `find_near_duplicates` to drop the SECOND document in every near-duplicate pair (keeping the first), returning a cleaned corpus dict.

**Exercise 3 (Extend):** Notebook 33 built CI regression gates. Sketch a data-quality gate that would run in CI whenever the RAG corpus changes — what should it check using `find_near_duplicates` and `chunk_quality_check` before allowing a corpus update to merge?


In [ ]:
# Exercise 1: Warm-up
# Task: Add a corpus_v3 that removes doc1, sync it, and confirm 'deleted' reports doc1.
# Hint: corpus_v3 = {k: v for k, v in corpus_v2.items() if k != "doc1"}

# YOUR CODE HERE


# Exercise 2: Apply
# Task: Implement dedup_corpus(documents, threshold) -> dict, dropping the second of each pair.
# Hint: find_near_duplicates returns (id_a, id_b, ratio) tuples; collect id_b values to drop.

# YOUR CODE HERE


# Exercise 3: Extend
# Task: Sketch a CI data-quality gate for RAG corpus changes.
# Hint: think about what should HARD BLOCK (e.g. any chunk failing chunk_quality_check) versus
# WARN (e.g. a near-duplicate ratio just above threshold that might be intentional).

# YOUR CODE HERE


<details>
<summary>Click to reveal solutions</summary>

```python
# Exercise 1
corpus_v3 = {k: v for k, v in corpus_v2.items() if k != "doc1"}
to_embed, unchanged, deleted = index.sync(corpus_v3)
print(f"Third sync: embed={to_embed}, unchanged={unchanged}, deleted={deleted}")
assert "doc1" in deleted

# Exercise 2
def dedup_corpus(documents, threshold=0.85):
    dupes = find_near_duplicates(documents, threshold)
    to_drop = {b for _, b, _ in dupes}
    return {k: v for k, v in documents.items() if k not in to_drop}

print(dedup_corpus(messy_corpus))

# Exercise 3
# A CI data-quality gate on a corpus-update PR:
#   HARD BLOCK: any chunk failing chunk_quality_check's length checks (too short/long is an
#   objective defect, always worth fixing before merge).
#   WARN (post a comment, don't block): near-duplicates above threshold, since some
#   near-duplication is intentional (e.g. two docs covering closely related but distinct
#   topics) and a human reviewer should confirm before dropping content.
#   HARD BLOCK: any chunk that fails BOTH boundary checks (starts AND ends mid-sentence) —
#   a strong signal of a genuinely broken chunking pipeline, not a borderline judgment call.
```
</details>

## Key Takeaways
- Incremental embedding (content-hash-keyed sync) turns "re-embed everything" into "re-embed only what changed" — the same discipline as any incremental ETL pipeline, applied to a vector index.
- Data quality gates (near-duplicate detection, chunk boundary/length checks) belong BEFORE the vector store, not discovered later as "why does retrieval keep returning the same three documents."
- Synthetic data generation is a genuine data-engineering skill applied to a new source — but it inherits notebook 25's contamination discipline: always scan synthetic output against your real eval set before using it.
- Trace/conversation storage is an ordinary schema-design problem — model it for the queries you'll actually run (cost over time, latency percentiles, eval scores by prompt version), not just "log everything."
- This is the intersection where a Data Engineer's existing instincts — pipeline design, incremental processing, schema modeling, data quality gates — become a genuine competitive edge in AI engineering rather than a background to work around.

## What's Next
The Tier 6 capstones (P1, P2, P4) synthesize everything built across this curriculum into complete, evaluated, production-shaped systems.
